# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price'] #creating a new column on the existing data frame that will multiply the values of qty and price together to get the revenue value
total_revenue = df['revenue'].sum() #summing the revenue together within the revenue column will give us the total
total_units = df['qty'].sum() #adding together all of the quantities of each item will give us the total units

print(df)
print(f'Total revenue: ${total_revenue:.2f}') #makes a format to print whatever the total revenue variable is
print(f'Total units: {total_units}') #makes a format to print whatever the total revenue variable is
print(f'These answers mean that {total_units} items being sold ended up generating ${total_revenue:.2f} worth of revenue.') #sentence explaining answers

    vendor_id  category  qty  price  revenue
0        V-10     Drink    2   24.0     48.0
1        V-18  RainGear    1   12.0     12.0
2        V-18     Drink    3    4.5     13.5
3        V-10      Food    2   12.0     24.0
4        V-18     Drink    3    7.5     22.5
..        ...       ...  ...    ...      ...
395      V-18     Merch    1   12.0     12.0
396      V-01     Merch    2   24.0     48.0
397      V-10      Food    3    7.5     22.5
398      V-18     Merch    2   24.0     48.0
399      V-10     Drink    1    7.5      7.5

[400 rows x 5 columns]
Total revenue: $8520.00
Total units: 783
These answers mean that 783 items being sold ended up generating $8520.00 worth of revenue.


**Markdown:** Here I added another column, revenue, to the existing data frame as well as calculated total revenue and total units. I did this because it is useful to find how much you are making from each sale, not just how many objects you are selling in total. I was able to exhibit this by making the revenue column multiply the quantity and price values to produce revenue. Here I found that selling 783 units produced $8520 of revenue.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
by_category = df.groupby('category').agg(revenue = ('revenue','sum'), units = ('qty', 'sum'), orders = ('vendor_id', 'count')).sort_values('revenue', ascending = False)
#the row above manipulates the orginal table so that we get only the values that you are looking for -- first we group by category with each category getting the total revenue (sum of revenue), number of units sold (quantity sum) and total number of orders (vendor id summed since each vendor purchase is an order), not ascending would mean descending (highest to lowest) revenue

by_category['percent_share_of_total'] = (by_category['revenue']/total_revenue)*100 #puts the share of total as a percentage for each category
by_category['percent_share_of_total'] = by_category['percent_share_of_total'].round(1) #rounding percentage

print(by_category)


          revenue  units  orders  percent_share_of_total
category                                                
Food       4293.0    362     186                    50.4
Merch      1771.5    158      79                    20.8
Drink      1554.0    178      89                    18.2
RainGear    901.5     85      46                    10.6


**Answer:** This table shows that the category with the most revenue is food taking up a little over 50 percent of total revenue at 50.4%. However, as you can see by the table percentage share of revenue is not solely determined by number of orders or number of units, specifically looking at the merch and drink category. So what it reveals is that price is also a factor to consider for each category.

**Markdown:** I created a new table based on the information in the original data frame grouping by category so that I could see how the revenue would breakdown across each one. This was useful because it helped provide better analysis to revenue and in real life would help a business determine which categories are underperforming and may need help.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
vendor_report = (df.groupby('vendor_id').agg(average_order_revenue = ('revenue','mean'), order_count = ('revenue','size')).sort_values('average_order_revenue', ascending = False)).round(2)
#grouping the data by each vendor, then taking the averages for the vendor's revenue, reporting the size of revenue which would be how many line contributed to revenue so that would be order number, then sorting from highest to lowest

print(vendor_report)
top_vendor = vendor_report.index[0] #since we sorted in descending order, the person at the top of the list (index 0) would have the highest average order revenue
print(f'The vendor with the highest average order revenue is {top_vendor} with an average of ${vendor_report.iloc[0]['average_order_revenue']} over {vendor_report.iloc[0]['order_count']} orders.')

           average_order_revenue  order_count
vendor_id                                    
V-01                       22.60           94
V-18                       21.75          108
V-05                       20.58           93
V-10                       20.31          105
The vendor with the highest average order revenue is V-01 with an average of $22.6 over 94.0 orders.


**Answer:** Vendor V-01 has the highest average order revenue with $22.60 per order over 94 orders. This can make the data slightly skewed since some of the other vendors have sold more items so their averages will be lower, since they are spread out across more quantity.

**Markdown:** Here I divided up the revenue by both vendor and over orders per vendor. We did this because the ratio can tell you something about profitability. I was able to do this by grouping on vendor and splitting up revenue and orders accordingly.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
merch_share = ((df.loc[df['category']=='Merch', 'revenue'].sum()) / total_revenue) * 100 #locating the category that is called Merch and then taking its correpsonding revenue value to divide by the total revenue and get the percentage share
print(f'The share of revenue that comes from Merch is {merch_share:.1f}% of the overall ${total_revenue:.2f} total revenue.')

The share of revenue that comes from Merch is 20.8% of the overall $8520.00 total revenue.


**Answer:** The Merch category accounts for 20.8% of the total $8520 revenue produced.

**Markdown:** I took the revenue from merch and divided it over total revenue to see that percentage share that it has in the revenue across all categories. This is good because it reveals which categories are succeeding in getting sales and money.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
rows_before = len(df) #counts the rows in the df table
revenue_before = df['revenue'].sum() #adding together all the revenue in the original table

joined = df.merge(vendor_names, on = 'vendor_id', how = 'left', validate = 'many_to_one') #merges the two tables using the vendor_names df on the vendor_id with the requirements of left join and validate

rows_after = len(joined) #counts the rows in the table
revenue_after = joined['revenue'].sum() #adding together all the revenue in the table

print(f'Rows before merge: {rows_before}') #prints rows before
print(f'Rows after merge: {rows_after}') #prints rows after
print(f'Revenue before merge: {revenue_before}') #prints revenue before
print(f'Revenue after merge: {revenue_after}') #prints revenue after

unmatched = joined.loc[joined['vendor_name'].isna(), 'vendor_id'].iloc[0] #tells you the vendor id that is missing a corresponding vendor name
print(f'Unmatched vendor id is: {unmatched}')
joined['vendor_name'] = joined['vendor_name'].fillna('Unmatched vendor') #finds the vendor name that is missing a value and replaced it with 'Unmatched vendor'
print(joined)


Rows before merge: 400
Rows after merge: 400
Revenue before merge: 8520.0
Revenue after merge: 8520.0
Unmatched vendor id is: V-18
    vendor_id  category  qty  price  revenue       vendor_name
0        V-10     Drink    2   24.0     48.0   Cav Merch North
1        V-18  RainGear    1   12.0     12.0  Unmatched vendor
2        V-18     Drink    3    4.5     13.5  Unmatched vendor
3        V-10      Food    2   12.0     24.0   Cav Merch North
4        V-18     Drink    3    7.5     22.5  Unmatched vendor
..        ...       ...  ...    ...      ...               ...
395      V-18     Merch    1   12.0     12.0  Unmatched vendor
396      V-01     Merch    2   24.0     48.0      Hoos Burgers
397      V-10      Food    3    7.5     22.5   Cav Merch North
398      V-18     Merch    2   24.0     48.0  Unmatched vendor
399      V-10     Drink    1    7.5      7.5   Cav Merch North

[400 rows x 6 columns]


**The unmatched vendor, and what I did about it:** The unmatched vendor was vendor V-18. I decided to change this vendor's vendor name to "Unmatched vendor" as opposed to dropping the vendor values as a whole because I didn't wan to skew the data. By making it recognized that the particular vendor is unmatched, there can then be more research into who the vendor is, completing the data. This is better than just fully dropping the category because then we would lose some of the revenue values and that would skew the percentage shares of each category and other information.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
pivot = pd.pivot_table(df, index = 'vendor_id', columns = 'category', values = 'revenue',aggfunc = 'sum',margins = True, margins_name = 'Total') #use pandas pivot table function and instructions to fill in the table
#use the vendor id as a index so it goes as the value down the side, columns are each category, values in the colums are revenue, and we want to use sum to give us totals
print(pivot)

category    Drink    Food   Merch  RainGear   Total
vendor_id                                          
V-01        171.0  1338.0   373.5     241.5  2124.0
V-05        298.5   882.0   489.0     244.5  1914.0
V-10        502.5  1054.5   400.5     175.5  2133.0
V-18        582.0  1018.5   508.5     240.0  2349.0
Total      1554.0  4293.0  1771.5     901.5  8520.0


**Answer:** The table breaks down vendor revenue across categories as well as compared to each other vendor. For example we can see that vendor V-01 made \$2124.00 revenue of the total \$8520.00 and drinks accounted for \$171, food for \$1338, merch for \$373.50, and rain gear for \$241.50.

**Markdown:** Here I created a pivot table with the data frame information we have. This is helpful because we can pick and choose the information we want and put it in an easier to view format such as the pivot table here which looks more like a spreadsheet.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')
#I went back through the previous questions to change my variable names to match the assert statements so that they would run properly

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

**Answer:**                                                 
**a)** I would tell the vendors that since the Food category generated over half of the revenue with \$4293.00 they should ensure that this is always well stocked in their inventories. I would also tell them to analyze their competitors' pricing since looking at average order revenue reveals that even the companies selling more aren't necessarily making more. For example vendor V-01 had an average order revenue of \$22.60 over 94 orders while vendor
V-18 had an avergae order revenue of $21.75 for 108 orders. This illustrates that it was likely a pricing difference between the two that gave vendor V-18 the competitive edge. We can see it even farther in the original data given where V-10	sells 2 Drinks	for	\$24.0 each while V-18 sells 3 drinks for \$4.5 each. Overall, the vendors should change their pricing to be competitive with the other vendors, and ensure that a large portion of their business is devoted to food in order to capture the most revenue.

**b)** I think question 3 has the least trustworthy answer due to varying group size. Despite looking at average revenue per order, each order is comprised of a different amount of items within the order. This means that we can't truly see which vendor is making the most money or not because one vendor's order could have 12 items while another's only has 1. Additionally the fact that not all of the vendors had the same order count or same order prices meant that the comparison was not even. V-01 had the highest average order revenue but V-18 was the vendor with the most orders.